In [ ]:
# Databricks notebook source
import requests
import json
import time

import pyspark.sql.functions as F
from delta.tables import DeltaTable

# COMMAND ----------

dbutils.widgets.text("workspace_url", spark.conf.get("spark.databricks.workspaceUrl"))
dbutils.widgets.text("output_catalog", "poc")
dbutils.widgets.text("output_schema", "testing")

In [ ]:
# COMMAND ----------

# store service principal token in secret scope, retrieve from there
access_token = dbutils.secrets.get(scope = "my_scope", key = "my_key")

In [ ]:
# COMMAND ----------

workspace_url = dbutils.widgets.get("workspace_url")
output_catalog = dbutils.widgets.get("output_catalog")
output_schema = dbutils.widgets.get("output_schema")
job_runs_table = f"{output_catalog}.{output_schema}.job_runs"

print(job_runs_table)

IllegalArgumentException : Secret does not exist with scope: and key:

Error: [0;31m---------------------------------------------------------------------------[0m
[0;31mIllegalArgumentException[0m                  Traceback (most recent call last)
File [0;32m~/.ipykernel/1546/command--1-3284167437:4[0m
[1;32m      1[0m [38;5;66;03m# COMMAND ----------[39;00m
[1;32m      2[0m 
[1;32m      3[0m [38;5;66;03m# store service principal token in secret scope, retrieve from there[39;00m
[0;32m----> 4[0m access_token [38;5;241m=[39m dbutils[38;5;241m.[39msecrets[38;5;241m.[39mget(scope [38;5;241m=[39m [38;5;124m"[39m[38;5;124m"[39m, key [38;5;241m=[39m [38;5;124m"[39m[38;5;124m"[39m) 
[1;32m      6[0m [38;5;66;03m# COMMAND ----------[39;00m
[1;32m      8[0m workspace_url [38;5;241m=[39m dbutils[38;5;241m.[39mwidgets[38;5;241m.[39mget([38;5;124m"[39m[38;5;124mworkspace_url[39m[38;5;124m"[39m)

File [0;32m/databricks/python_shell/lib/dbruntime/dbutils.py:320[0m, in [0;36mDBUtils.SecretsHandler.get[0;34m(self,

In [ ]:
# COMMAND ----------

spark.sql(
    f"""
    CREATE TABLE IF NOT EXISTS {job_runs_table} (
        run_id STRING,
        job_id STRING,
        workspace_url STRING,
        job_run_json STRING,
        job_run VARIANT,
        content_hash STRING,
        updated_ts TIMESTAMP
    )
    CLUSTER BY (run_id, job_id)
    TBLPROPERTIES (
        delta.enableChangeDataFeed = true,
        delta.enableDeletionVectors = true
        )
    """
)


In [ ]:
# COMMAND ----------

def job_run_api(access_token, max_retries = 600):
    """ 
    Returns UDF for job runs API requests.
    """

    @F.udf("string")
    def job_run_api_udf(workspace_url, run_id):

        url = f"https://f{workspace_url}/api/2.2/jobs/runs/get?run_id={run_id}"
        headers = {
            "Authorization": f"Bearer {access_token}"
        }
        
        retries = 0

        while retries < max_retries:

            response = requests.get(url, headers=headers)
            
            if response.status_code == 200:
                return response.text
            
            retries += 1
            time.sleep(1)

        raise ConnectionError("")

    return job_run_api_udf


In [ ]:
# COMMAND ----------

# Define the API endpoint and headers
url = f"https://f{workspace_url}/api/2.2/jobs/runs/list"
headers = {
    "Authorization": f"Bearer {access_token}"
}
params = {
    "page_token": None
}

next_url = url
has_more = True
runs_list = []

# Query API and create a list of available job runs
while has_more:

    # Make the API request
    response = requests.get(next_url, headers=headers, params=params)
    if response.status_code != 200:
        raise ConnectionError("API request failed with status code {response.status}")
    response_json = response.json()

    # Create list of runs with JSON string
    runs = response.json()["runs"]
    runs_list += [
        {"run_id": run["run_id"], "job_id": run["job_id"]} for run in runs
        ]

    # Set URL to query next page
    params["page_token"] = response_json.get("next_page_token", None)
    has_more = params["page_token"] is not None

In [ ]:
# Create dataframe with job run response and variant column
df = spark.createDataFrame(runs_list)
df = df.withColumn("workspace_url", F.lit(workspace_url))
df = df.withColumn("job_run_json", job_run_api(access_token)(df.workspace_url, df.run_id))
df = df.withColumn("job_run", F.try_parse_json(df.job_run_json))
df = df.withColumn("content_hash", F.sha2(df.job_run_json, 256))
df = df.withColumn("updated_ts", F.current_timestamp())

# Execute merge into output table
delta_output_table = DeltaTable.forName(spark, job_runs_table)

(
    delta_output_table
        .alias('target')
        .merge(
            df.alias('updates'),
            'target.run_id = updates.run_id'
            )
        # Update only when content has changed
        .whenMatchedUpdateAll("target.content_hash != updates.content_hash")
        .whenNotMatchedInsertAll()
        .execute()
        .display()
)